## 1. Your first request
`requests.get(url)` goes to the web address you give it and downloads the page.
It gives you back a **response**: the response tells you if the download worked, and it contains the page itself.

- 🌐 https://quotes.toscrape.com/

> 💼 Every scraper (price tracker, job collector, news reader) starts with this one line.

In [1]:
import requests

In [2]:
response = requests.get("https://quotes.toscrape.com/", timeout=10)
response

<Response [200]>

`<Response [200]>` means: it worked.

## 2. What is inside the response?
The response contains more than the page. It also tells you if the request worked,
the address you finally arrived at, and what kind of content you received (HTML, JSON, an image...).

- 🌐 On https://quotes.toscrape.com/ press `Ctrl+U` to see the HTML of the page. `response.text` contains exactly this text.

> 💼 Check `Content-Type` first: a program that expects HTML and gets an image will crash.

In [3]:
print("status code :", response.status_code)
print("ok?         :", response.ok)
print("final URL   :", response.url)
print("content type:", response.headers["Content-Type"])
print("encoding    :", response.encoding)

status code : 200
ok?         : True
final URL   : https://quotes.toscrape.com/
content type: text/html; charset=utf-8
encoding    : utf-8


In [4]:
print(response.text[:500])   # the first 500 characters of the HTML

<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla


| Attribute | You get |
|---|---|
| `response.text` | the page as **text** (HTML) |
| `response.content` | the page as **bytes** (for images, PDF...) |
| `response.json()` | the page as a **Python dict** (only if it is JSON) |

In [5]:
print(type(response.text), len(response.text))
print(type(response.content), len(response.content))

<class 'str'> 11021
<class 'bytes'> 11064


## 3. Status codes
Every response comes with a number: the **status code**.
It tells you if the server sent you the page, or why it did not.

| Code | Meaning |
|---|---|
| 200 | OK |
| 301 / 302 | moved somewhere else (requests follows it for you) |
| 403 | forbidden: the site refuses you |
| 404 | page not found |
| 429 | too many requests: slow down |
| 500 | the server has a problem |

- 🌐 Open [httpbin.org/status/404](https://httpbin.org/status/404), press `F12` → **Network** → reload: the **Status** column says `404`.
  Your browser also receives a status code, it just does not show it to you.

> 💼 You check 1,000 product pages: 404 = product removed, 429 = slow down, 500 = try later.

In [6]:
for code in [200, 404, 500]:
    r = requests.get(f"https://httpbin.org/status/{code}", timeout=10)
    print(code, "->", r.status_code, "ok =", r.ok)

200 -> 200 ok = True
404 -> 404 ok = False
500 -> 500 ok = False


### Redirects
Sometimes a page has moved to a new address. When you ask for the old address,
the server answers: *"this page is now at another address, go there"*. This is called a **redirect**.

requests goes to the new address automatically, so you always receive the final page.
If you want to see every address it passed through on the way, look at `response.history`.

- 🌐 Open [httpbin.org/redirect/2](https://httpbin.org/redirect/2): you asked for `/redirect/2`, but the address bar now shows `/get`.
  Your browser was redirected too.

> 💼 Short links (`bit.ly/...`) send you to another page: `r.url` tells you where you **really** arrived.

In [7]:
r = requests.get("https://httpbin.org/redirect/2", timeout=10)
print(r.status_code, r.url)
print(r.history)

200 https://httpbin.org/get
[<Response [302]>, <Response [302]>]


## 4. Query parameters
Many web addresses have a part after the `?`, for example `...?q=web+scraping&page=2`.
This part is a list of **name = value** pairs: here `q` is `web scraping` and `page` is `2`.
The website reads these pairs to know what you are looking for.

In Python you do not type this part by hand. You write the pairs in a dictionary,
`{"q": "web scraping", "page": 2}`, and give it to `requests.get(url, params=...)`.
requests adds the `?...` part to the address for you, and downloads the page.

- 🌐 Search *web scraping* on Google and look at the address bar: [google.com/search?q=web+scraping](https://www.google.com/search?q=web+scraping).
  Your search words are in the part after `?`.

> 💼 Search words, filters and page numbers are in this part: change `page` in a loop to get page 1, 2, 3...

In [8]:
params = {"q": "web scraping", "page": 2}
r = requests.get("https://httpbin.org/get", params=params, timeout=10)
print(r.url)

https://httpbin.org/get?q=web+scraping&page=2


Open the address that was printed: https://httpbin.org/get?q=web+scraping&page=2

## 5. JSON answers
Some addresses do not send a web page: they send **JSON**, which is plain data written as names and values.
`.json()` turns this data into a Python dictionary, so you can read each value by its name.

- 🌐 The page you just opened is JSON: only `{ }`, names and values.

> 💼 Weather apps, maps, bank apps: they all get JSON from an API. No HTML to read.

In [9]:
data = r.json()
print(type(data))
data["args"]

<class 'dict'>


{'page': '2', 'q': 'web scraping'}

In [10]:
data["headers"]["User-Agent"]

'python-requests/2.34.2'

By default you tell every site *"I am python-requests"*. Some sites block that.

## 6. Headers: say who you are
When your program asks for a page, it also sends some extra information about itself: the **headers**.
The most important one is the **User-Agent**: it tells the website who is asking (a browser, a program...).

- 🌐 Open https://httpbin.org/headers this page shows the headers your browser sent.
  Its User-Agent (`Mozilla/5.0 ...`) is your browser saying who it is.

> 💼 Some sites refuse programs without a name (Wikipedia answers 403 in notebook 2). Send a name and an email.

In [11]:
headers = {
    "User-Agent": "ESILV-Lab/1.0 (educational use; contact: student@example.com)",
    "Accept-Language": "en-US,en;q=0.9",
}
r = requests.get("https://httpbin.org/headers", headers=headers, timeout=10)
r.json()["headers"]

{'Accept': '*/*',
 'Accept-Encoding': 'gzip, deflate, br, zstd',
 'Accept-Language': 'en-US,en;q=0.9',
 'Host': 'httpbin.org',
 'User-Agent': 'ESILV-Lab/1.0 (educational use; contact: student@example.com)'}

The server also sends headers back:

In [12]:
for name, value in response.headers.items():
    print(f"{name:25} {value}")

Date                      Mon, 23 Nov 2026 15:58:46 GMT
Content-Type              text/html; charset=utf-8
Transfer-Encoding         chunked
Connection                keep-alive
Strict-Transport-Security max-age=0; includeSubDomains; preload
Content-Encoding          br


## 7. Always set a timeout
If a website never answers, your program waits for it **forever**.
`timeout=10` means: wait at most 10 seconds, then stop and raise an error.

- 🌐 https://httpbin.org/delay/3 waits 3 seconds before it answers. The code below waits only 1 second, so it gives up.

> 💼 One dead site out of 500 must not block your program all night.

In [13]:
try:
    requests.get("https://httpbin.org/delay/3", timeout=1)
except requests.exceptions.Timeout:
    print("Too slow: gave up after 1 second")

Too slow: gave up after 1 second


## 8. Handle errors
When a page is not found (404), Python does **not** stop: the response simply has status 404,
and your code continues as if nothing happened.

If you want a real Python error for every bad status (4xx or 5xx), call `response.raise_for_status()`.
Then you can catch that error with `try` / `except`.

> 💼 One broken page out of 500: write it in an error list and go on.

In [14]:
r = requests.get("https://httpbin.org/status/404", timeout=10)
print(r.status_code)

try:
    r.raise_for_status()
except requests.exceptions.HTTPError as error:
    print("HTTP error:", error)

404
HTTP error: 404 Client Error: Not Found for url: https://httpbin.org/status/404


Here is a small function you can reuse. `RequestException` catches **every** kind of requests problem:
a bad status, a timeout, no internet, or a wrong address.

In [15]:
def fetch(url, **options):
    """Return the response, or None if something went wrong."""
    try:
        r = requests.get(url, timeout=10, **options)
        r.raise_for_status()
        return r
    except requests.exceptions.RequestException as error:
        print("Failed:", error)
        return None

We test it three times: a good page, a page that does not exist, and a website that does not exist.

In [16]:
print(fetch("https://quotes.toscrape.com/"))
print(fetch("https://httpbin.org/status/404"))
print(fetch("https://this-site-does-not-exist.example/"))

<Response [200]>
Failed: 404 Client Error: Not Found for url: https://httpbin.org/status/404
None
Failed: HTTPSConnectionPool(host='this-site-does-not-exist.example', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='this-site-does-not-exist.example', port=443): Failed to resolve 'this-site-does-not-exist.example' ([Errno 8] nodename nor servname provided, or not known)"))
None


## 9. Sending data: POST
Until now we only **read** pages: this is a `GET` request.
When you fill in a form and click the button (to log in, or to search), your browser **sends** data to the site:
this is a `POST` request. `requests.post()` does the same thing from Python.

- 🌐 Fill in the form at https://httpbin.org/forms/post and click the button at the bottom.
  httpbin shows you the data your browser sent.

> 💼 Some data appears only after you log in or send a search form.

In [17]:
form = {"username": "alice", "password": "secret"}
r = requests.post("https://httpbin.org/post", data=form, timeout=10)
r.json()["form"]

{'password': 'secret', 'username': 'alice'}

In [18]:
r = requests.post("https://httpbin.org/post", json={"symbol": "AAPL", "days": 5}, timeout=10)
r.json()["json"]

{'days': 5, 'symbol': 'AAPL'}

`data=` sends a form. `json=` sends JSON.

## 10. Sessions: keep cookies and settings
Websites give your browser small pieces of information called **cookies** (your language, your login...).
Your browser keeps them and sends them back every time it visits the site.

A plain `requests.get()` forgets the cookies after each request.
A `Session` keeps them, like a browser does. It also keeps your headers, so you set them only once.

- 🌐 Open [httpbin.org/cookies/set/lang/fr](https://httpbin.org/cookies/set/lang/fr): the site gives your browser a cookie `lang=fr`.
  Then open [httpbin.org/cookies](https://httpbin.org/cookies): your browser still has it.

> 💼 After you log in, the login cookie keeps you logged in. Without a Session, you are logged out again.

In [19]:
session = requests.Session()
session.headers.update(headers)                                 # every request sends our User-Agent

session.get("https://httpbin.org/cookies/set/lang/fr", timeout=10)  # the server gives us a cookie
session.get("https://httpbin.org/cookies", timeout=10).json()       # the session sends it back

{'cookies': {'lang': 'fr'}}

In [20]:
requests.get("https://httpbin.org/cookies", timeout=10).json()      # without a session: no cookie

{'cookies': {}}

## 11. Be polite: wait between requests
A program can ask for pages much faster than a person can read them. Too many fast requests slow the website down.
So we wait 1 second between two pages, with `time.sleep(1)`.

- 🌐 On [quotes.toscrape.com](https://quotes.toscrape.com/) click **Next →**: the address becomes [quotes.toscrape.com/page/2/](https://quotes.toscrape.com/page/2/).
  Only the number changes, so a loop over 1, 2, 3 visits every page.

> 💼 100 requests per second slows the site down for real people, and gets you blocked.

In [21]:
import time

for page in range(1, 4):
    r = session.get(f"https://quotes.toscrape.com/page/{page}/", timeout=10)
    print("page", page, "| status", r.status_code, "|", r.text.count('class="quote"'), "quotes")
    time.sleep(1)   # wait 1 second before the next page

page 1 | status 200 | 10 quotes
page 2 | status 200 | 10 quotes
page 3 | status 200 | 10 quotes


## 12. 200 is not always success
A 200 only means *"the server answered"*. Some APIs say *"not found"* **inside** the answer, with a 200.

- 🌐 Open [this Open Food Facts address](https://world.openfoodfacts.org/api/v2/product/0000000000000) (a barcode that does not exist): read the answer, there is no product.

> 💼 Check the content too, not only the status code. Otherwise you save empty rows without noticing.

In [22]:
response = requests.get("https://world.openfoodfacts.org/api/v2/product/0000000000000",
                        headers={"User-Agent": "ESILV-Lab/1.0 (educational use; student@example.com)"}, timeout=20)
data = response.json()
print("status code:", response.status_code)                      # 200 ...
print("found?     :", data["status"], data["status_verbose"])     # ... but 0 = not found

status code: 200
found?     : 0 no code or invalid code


## 13. Retries: try again, a few times
Some errors are **temporary**: 429 (slow down) or 503 (server busy). Waiting and trying again often works.

`Retry` + `HTTPAdapter` do it for you, inside a session: at most 3 more tries, waiting a bit longer each time.
Only for errors that can get better (429, 5xx). Never for 400 or 404: they will not change.

- 🌐 https://httpbin.org/status/503 always answers 503, so our session tries 3 more times, then gives up.

> 💼 Your program downloads 5,000 pages at night. The server is busy for 2 seconds: with retries, nothing is lost.

In [23]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

retry = Retry(total=3,                                   # at most 3 more tries
              backoff_factor=0.5,                        # wait 0 s, 1 s, 2 s between them
              status_forcelist=[429, 500, 502, 503, 504],  # only for these errors
              allowed_methods=["GET"],                   # only for GET (safe to repeat)
              respect_retry_after_header=True)           # if the server says how long to wait, wait that long
retry_session = requests.Session()
retry_session.mount("https://", HTTPAdapter(max_retries=retry))

start = time.time()
try:
    retry_session.get("https://httpbin.org/status/503", timeout=10)
except requests.exceptions.RetryError:
    print("gave up after 3 retries")
print(f"{time.time() - start:.1f} seconds")

gave up after 3 retries
3.6 seconds


## 14. Prepared requests: check before you send
Build the request, **look at it**, and send it only when you are sure.
Do this before any request that **changes** something on a website (a POST that writes data).
`auth=("user", "password")` adds a login to the request.

> 💼 A wrong POST can change real data. Check it first, then send it.

In [24]:
prepared = requests.Request("POST", "https://httpbin.org/post",
                            data={"code": "123", "quantity": "400 g"},
                            auth=("user", "secret")).prepare()      # built, NOT sent
print(prepared.method, prepared.url)
print(prepared.headers)
print(prepared.body)
# when you are sure: requests.Session().send(prepared, timeout=10)

POST https://httpbin.org/post
{'Content-Length': '23', 'Content-Type': 'application/x-www-form-urlencoded', 'Authorization': 'Basic dXNlcjpzZWNyZXQ='}
code=123&quantity=400+g


## 15. Big downloads: streaming
`response.content` keeps the **whole** file in memory. For a big file, use `stream=True`
and write it piece by piece with `iter_content()`.

> 💼 A 2 GB dataset does not fit in memory. Streaming writes it 64 KB at a time.

In [25]:
from pathlib import Path

Path("output").mkdir(exist_ok=True)
written = 0
with requests.get("https://httpbin.org/bytes/200000", stream=True, timeout=10) as response:
    response.raise_for_status()
    with open("output/download.bin", "wb") as file:
        for chunk in response.iter_content(chunk_size=64 * 1024):   # 64 KB at a time
            file.write(chunk)
            written += len(chunk)
print(written, "bytes written")

200000 bytes written
